In [ ]:
# main.py
import sys
sys.path.append('../research')   # add the research folder to Python’s import path
import pandas as pd
from Data_Preprocessing import DataPreprocessingPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
import numpy as np
from lightgbm import LGBMClassifier
# Load your dataset
df = pd.read_csv('/home/xaris/Desktop/Projects/Introvert & Extrovert/Introvert-vs-Extrovert/Datasets/train.csv')

# Define columns
numerical = ['Time_spent_Alone', 'Social_event_attendance','Going_outside','Friends_circle_size','Post_frequency']
categorical = ['Stage_fear','Drained_after_socializing']
target = ['Personality']

# Initialize and run preprocessing
pipeline = DataPreprocessingPipeline(df, numerical, categorical)
df_processed = pipeline.run_pipeline()

# Check results
print(df_processed.head())

   Time_spent_Alone Stage_fear  Social_event_attendance  Going_outside  \
0               0.0         No                      6.0            4.0   
1               1.0         No                      7.0            3.0   
2               6.0        Yes                      1.0            0.0   
3               3.0         No                      7.0            3.0   
4               1.0         No                      4.0            4.0   

  Drained_after_socializing  Friends_circle_size  Post_frequency  Personality  
0                        No                 15.0        5.000000            1  
1                        No                 10.0        8.000000            1  
2                       NaN                  3.0        0.000000            0  
3                        No                 11.0        5.000000            1  
4                        No                 13.0        6.113682            1  


In [2]:
df_processed.shape

(18524, 8)

In [3]:
df_processed.head()

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0.0,No,6.0,4.0,No,15.0,5.000000,1
1,1.0,No,7.0,3.0,No,10.0,8.000000,1
2,6.0,Yes,1.0,0.0,NaN,3.0,0.000000,0
3,3.0,No,7.0,3.0,No,11.0,5.000000,1
4,1.0,No,4.0,4.0,No,13.0,6.113682,1


In [4]:
df_processed.isnull().sum()

Time_spent_Alone                0
Stage_fear                   1893
Social_event_attendance         0
Going_outside                   0
Drained_after_socializing    1149
Friends_circle_size             0
Post_frequency                  0
Personality                     0
dtype: int64

In [ ]:
X = df.drop(['Personality'],axis=1)
y = df[['Personality']]

# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((14819, 8), (3705, 8))

In [8]:
# check the the unique values for target column (Personality)
unique, count = np.unique(y_train, return_counts = True)
Y_train_dict_value_count = {k:v for (k,v) in zip(unique, count)}
Y_train_dict_value_count

{'Extrovert': np.int64(10946), 'Introvert': np.int64(3873)}

So we should implement an oversample technique to fix the imbalanced data.<br>
Specifically we will use SMOTE (Synthetic Minority Over-sampling Technique).

In [ ]:
# One hot encode the "Stage_fear" and "Drained_after_socializing" and standard scale on numeric features
#numeric_transformer = StandardScaler()
#oh_transformer = OneHotEncoder()
#categorical_imputer = SimpleImputer(strategy='most_frequent')

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Standard Scale for the numerical fetures
numerical_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=
    [
        ("categorical",categorical_pipeline,categorical),
        ("numerical", numerical_pipeline, numerical), 
    ]
)

pipeline = Pipeline(steps=[('preprocessor',preprocessor),
                            ('smote'),SMOTE(random_state=42)])

In [ ]:
# Models i 'll use to classify the data
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Gradient Boost": GradientBoostingClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(), 
    'lightgbm':LGBMClassifier()
}

In [ ]:
# Create param grids for every and each of these algorithms
# Decision Tree Hyperparameters
DT_params = {  
    'criterion':['gini', 'entropy'],
    'splitter':'best',
    'max_depth':[2, 3, 5, 10, 20],
    'min_samples_split':[5, 10, 20, 50, 100],
    'min_samples_leaf':[2,3,4,5,6,7,8,9]
}

# Random Forest Hyperparameters
RF_params = {
    'n_estimators':[100, 200],
    'max_depth':[2, 3, 5, 10, 20],
    'min_samples_split':[5, 10, 20, 50, 100],
    'min_samples_leaf':[2,3,4,5,6,7,8,9],
    'bootstrap': [True, False]
    }

# AdaBoost hyperparameters
Ada_params = {
    'n_estimators':[100, 200],
    'learning_rate':[0.01,0.1,0.2]
}

# Gradient Boosthyperparameters
GradB_params = {
    'learning_rate':[0.01,0.1,0.2],
    'n_estimators':[100, 200],
    'min_samples_split':[5, 10, 20, 50, 100],
    'min_samples_leaf':[2,3,4,5,6,7,8,9],
}

# SVM hyperparameters
SVM_params = {
    'C': [0.1, 1, 10, 100, 1000], # regularization parameter
    'kernel':['linear','poly','sigmoid'],
    'gamma': [1, 0.1, 0.01, 0.001, 0.0001]
}

# KNN hyperparameters
KNN_params = {
    'n_neighbors':range(1, 21, 2),
    'weights' : ['uniform', 'distance'],
    'metric' : ['euclidean', 'manhattan', 'minkowski']
}

# lightgbm hyperparameters
lgb_params = {
    'learning_rate':[0.01,0.1,0.2],
    'max_depth':[2, 3, 5, 10, 20],
    'min_data_in_leaf':[2,3,4,5,6,7,8,9],
}